# Order book tutorial

A small, runnable introduction to the core `OrderBook` operations. Each code cell prints what it is doing before showing the result. Run the cells from top to bottom.

Prices are integer ticks and quantities are integer lots. A **bid** is a buy order; an **ask** is a sell order.

In [1]:
from pprint import pprint

from order_book import OrderBook

print("Imported OrderBook: it stores orders, matches trades, and reports the book state.")

Imported OrderBook: it stores orders, matches trades, and reports the book state.


## 1. Create a book

The constructor takes the number of price levels to keep, a starting mid-price, and the distance from that mid-price used to seed the bid and ask anchors. An empty book has no actual quotes yet.

In [2]:
print("OrderBook(...): create an empty book with a 10-tick window around mid-price 10,000.")
book = OrderBook(num_levels=10, mid_price=10_000, distance_from_mid=5)
print("Created book. There are no resting orders yet:")
print("  best bid:", book.best_bid)
print("  best ask:", book.best_ask)
print("  mid-price:", book.mid_price)
print("  spread:", book.spread)

OrderBook(...): create an empty book with a 10-tick window around mid-price 10,000.
Created book. There are no resting orders yet:
  best bid: None
  best ask: None
  mid-price: None
  spread: None


## 2. Understand the price window

`anchor(side)` gives the reference price for one side; `edge(side)` gives its far boundary. `bid_prices` and `ask_prices` expose those two numbers as arrays. These are allowed-window boundaries, **not** quotes in the book.

In [3]:
print("anchor(side): get the current reference price; edge(side): get the far window boundary.")
print("  bid anchor / edge:", book.anchor("bid"), book.edge("bid"))
print("  ask anchor / edge:", book.anchor("ask"), book.edge("ask"))
print("bid_prices and ask_prices: view each side's [anchor, edge] window.")
print("  bid window:", book.bid_prices)
print("  ask window:", book.ask_prices)

anchor(side): get the current reference price; edge(side): get the far window boundary.
  bid anchor / edge: 9995 9986
  ask anchor / edge: 10005 10014
bid_prices and ask_prices: view each side's [anchor, edge] window.
  bid window: [9995 9986]
  ask window: [10005 10014]


## 3. Add limit orders

A limit order sets a maximum buy price or minimum sell price. It trades against compatible resting orders if any exist; any unfilled quantity rests in the book. `limit()` returns a report. Here we add two asks at the same price (to show FIFO priority), one deeper ask, and a bid.

In [4]:
print("limit('ask', ...): place client 1's ask for 6 lots at 10,005.")
ask1 = book.limit("ask", client_id=1, price=10_005, quantity=6)
pprint(ask1)

print("\nlimit('ask', ...): place client 2's ask at the same price; it queues behind client 1.")
ask2 = book.limit("ask", client_id=2, price=10_005, quantity=3)
pprint(ask2)

print("\nlimit('ask', ...): add a deeper ask at 10,008.")
ask3 = book.limit("ask", client_id=3, price=10_008, quantity=4)
pprint(ask3)

print("\nlimit('bid', ...): add a bid at 9,995. It does not cross the asks.")
bid1 = book.limit("bid", client_id=10, price=9_995, quantity=5)
pprint(bid1)

limit('ask', ...): place client 1's ask for 6 lots at 10,005.
{'avg_fill_price': None,
 'cancelled_quantity': 0,
 'client_id': 1,
 'evicted_orders': [],
 'filled_quantity': 0,
 'order_id': 100000,
 'order_type': 'limit',
 'price': 10005,
 'replaced_order_id': None,
 'requested_quantity': 6,
 'resting_quantity': 6,
 'side': 'ask',
 'stp_cancelled_orders': [],
 'trades': []}

limit('ask', ...): place client 2's ask at the same price; it queues behind client 1.
{'avg_fill_price': None,
 'cancelled_quantity': 0,
 'client_id': 2,
 'evicted_orders': [],
 'filled_quantity': 0,
 'order_id': 100001,
 'order_type': 'limit',
 'price': 10005,
 'replaced_order_id': None,
 'requested_quantity': 3,
 'resting_quantity': 3,
 'side': 'ask',
 'stp_cancelled_orders': [],
 'trades': []}

limit('ask', ...): add a deeper ask at 10,008.
{'avg_fill_price': None,
 'cancelled_quantity': 0,
 'client_id': 3,
 'evicted_orders': [],
 'filled_quantity': 0,
 'order_id': 100002,
 'order_type': 'limit',
 'price': 10008,

## 4. Inspect the book

The best-price properties summarize the top of book. `levels(side)` lists price and total quantity, best price first. The lookup methods answer focused questions; `get_snapshot()` bundles common state into one dictionary.

In [5]:
print("best_bid, best_ask, mid_price, spread: read top-of-book properties.")
print("  best bid:", book.best_bid)
print("  best ask:", book.best_ask)
print("  mid-price:", book.mid_price)
print("  spread:", book.spread)

print("\nlevels(side): show aggregated price levels, best first.")
print("  bids:", book.levels("bid"))
print("  asks:", book.levels("ask"))

print("\nget_volume_at_price(side, price): total quantity at the ask price 10,005.")
print("  volume:", book.get_volume_at_price("ask", 10_005))

print("\nbest_excluding(side, order_id): find the best ask if ask1 is ignored.")
print("  best ask excluding ask1:", book.best_excluding("ask", ask1["order_id"]))

print("\nget_client_orders(client_id): list orders belonging to client 2.")
pprint(book.get_client_orders(2))

print("\nget_snapshot(): collect top-of-book, windows, levels, and total volume.")
pprint(book.get_snapshot())

best_bid, best_ask, mid_price, spread: read top-of-book properties.
  best bid: 9995
  best ask: 10005
  mid-price: 10000.0
  spread: 10

levels(side): show aggregated price levels, best first.
  bids: [(9995, 5)]
  asks: [(10005, 9), (10008, 4)]

get_volume_at_price(side, price): total quantity at the ask price 10,005.
  volume: 9

best_excluding(side, order_id): find the best ask if ask1 is ignored.
  best ask excluding ask1: 10005

get_client_orders(client_id): list orders belonging to client 2.
[{'client_id': 2,
  'order_id': 100001,
  'price': 10005,
  'quantity': 3,
  'side': 'ask'}]

get_snapshot(): collect top-of-book, windows, levels, and total volume.
{'ask_window': array([10005, 10014]),
 'asks': [(10005, 9, 2), (10008, 4, 1)],
 'best_ask': 10005,
 'best_bid': 9995,
 'bid_window': array([9995, 9986]),
 'bids': [(9995, 5, 1)],
 'mid_price': 10000.0,
 'spread': 10,
 'total_ask_volume': 13,
 'total_bid_volume': 5}


## 5. Submit a market order and see matching

A market order has no price limit: it consumes the best available opposing orders. Matching follows **best price first, FIFO within a price**, and trades execute at each resting order's price. Market orders are immediate-or-cancel, so any unfilled part is cancelled rather than added to the book.

`market()` calls the shared submission/matching logic internally (`_submit()` and `_match()`). Those underscored helpers are implementation details; normally call the public API.

In [6]:
print("market('bid', ...): client 99 buys 7 lots from the best asks.")
buy_report = book.market("bid", client_id=99, quantity=7)
pprint(buy_report)
print("\nThe first ask filled 6 lots; the next ask filled 1 lot. Inspect the remaining asks:")
print(book.levels("ask"))

market('bid', ...): client 99 buys 7 lots from the best asks.
{'avg_fill_price': 10005.0,
 'cancelled_quantity': 0,
 'client_id': 99,
 'evicted_orders': [],
 'filled_quantity': 7,
 'order_id': 100004,
 'order_type': 'market',
 'price': None,
 'replaced_order_id': None,
 'requested_quantity': 7,
 'resting_quantity': 0,
 'side': 'bid',
 'stp_cancelled_orders': [],
 'trades': [{'aggressor_side': 'buy',
             'buy_order_id': 100004,
             'buyer_id': 99,
             'price': 10005,
             'quantity': 6,
             'sell_order_id': 100000,
             'seller_id': 1,
             'trade_id': 1},
            {'aggressor_side': 'buy',
             'buy_order_id': 100004,
             'buyer_id': 99,
             'price': 10005,
             'quantity': 1,
             'sell_order_id': 100001,
             'seller_id': 2,
             'trade_id': 2}]}

The first ask filled 6 lots; the next ask filled 1 lot. Inspect the remaining asks:
[(10005, 2), (10008, 4)]


## 6. Look up and modify orders

`get_order(order_id)` returns one active order, or `None` if it is no longer active. Reducing an order's quantity with `modify_order()` keeps its ID and queue priority. Increasing quantity or changing price replaces it, so it gets a new ID and priority.

In [7]:
print("get_order(order_id): inspect the remaining bid.")
pprint(book.get_order(bid1["order_id"]))

print("\nmodify_order(..., new_quantity=3): reduce the bid from 5 to 3; its ID stays the same.")
reduced_id = book.modify_order(bid1["order_id"], new_quantity=3, client_id=10)
print("  returned ID:", reduced_id)
pprint(book.get_order(reduced_id))

print("\nmodify_order(..., new_price=9,996): change price; this replaces the order with a new ID.")
replaced_id = book.modify_order(reduced_id, new_price=9_996, client_id=10)
print("  old ID:", reduced_id, " new ID:", replaced_id)
print("last_order_report: inspect the most recent operation's report.")
pprint(book.last_order_report)
pprint(book.get_order(replaced_id))

get_order(order_id): inspect the remaining bid.
{'client_id': 10,
 'order_id': 100003,
 'price': 9995,
 'quantity': 5,
 'side': 'bid'}

modify_order(..., new_quantity=3): reduce the bid from 5 to 3; its ID stays the same.
  returned ID: 100003
{'client_id': 10,
 'order_id': 100003,
 'price': 9995,
 'quantity': 3,
 'side': 'bid'}

modify_order(..., new_price=9,996): change price; this replaces the order with a new ID.
  old ID: 100003  new ID: 100005
last_order_report: inspect the most recent operation's report.
{'avg_fill_price': None,
 'cancelled_quantity': 0,
 'client_id': 10,
 'evicted_orders': [],
 'filled_quantity': 0,
 'order_id': 100005,
 'order_type': 'limit',
 'price': 9996,
 'replaced_order_id': 100003,
 'requested_quantity': 3,
 'resting_quantity': 3,
 'side': 'bid',
 'stp_cancelled_orders': [],
 'trades': []}
{'client_id': 10,
 'order_id': 100005,
 'price': 9996,
 'quantity': 3,
 'side': 'bid'}


## 7. Cancel an order

`cancel_order(order_id, client_id=...)` removes that client's active order. Providing `client_id` checks ownership. The returned report tells you how much quantity was cancelled.

In [8]:
print("cancel_order(...): client 2 cancels the 2 lots remaining on ask2.")
cancel_report = book.cancel_order(ask2["order_id"], client_id=2)
pprint(cancel_report)
print("get_order(ask2 ID) now returns:", book.get_order(ask2["order_id"]))

cancel_order(...): client 2 cancels the 2 lots remaining on ask2.
{'avg_fill_price': None,
 'cancelled_quantity': 2,
 'client_id': 2,
 'evicted_orders': [],
 'filled_quantity': 0,
 'order_id': 100001,
 'order_type': 'cancel',
 'price': 10005,
 'replaced_order_id': None,
 'requested_quantity': 2,
 'resting_quantity': 0,
 'side': 'ask',
 'stp_cancelled_orders': [],
 'trades': []}
get_order(ask2 ID) now returns: None


## 8. Get array-form state

`get_book_arrays()` returns NumPy arrays for prices and volumes on a tick grid, useful for numerical code. `depth` controls how many ticks to include.

In [9]:
print("get_book_arrays(depth=5): get the first five ticks outward from each anchor.")
arrays = book.get_book_arrays(depth=5)
for name, array in arrays.items():
    print(f"  {name}: {array}")

get_book_arrays(depth=5): get the first five ticks outward from each anchor.
  bid_prices: [9996 9995 9994 9993 9992]
  bid_volumes: [3 0 0 0 0]
  bid_valid: [ True  True  True  True  True]
  ask_prices: [10008 10009 10010 10011 10012]
  ask_volumes: [4 0 0 0 0]
  ask_valid: [ True  True  True  True  True]


## 9. Self-trade prevention (optional)

By default, if an incoming order would trade with the same client's resting order, the incoming remainder is cancelled. This short example uses a fresh book so it doesn't affect the earlier walkthrough.

In [10]:
print("Create a fresh book, then use limit() to rest an ask owned by client 7.")
stp_book = OrderBook(num_levels=10, mid_price=10_000, distance_from_mid=5)
resting_ask = stp_book.limit("ask", client_id=7, price=10_005, quantity=2)

print("market('bid', client_id=7, ...): default self-trade prevention cancels the incoming buy.")
stp_report = stp_book.market("bid", client_id=7, quantity=1)
pprint(stp_report)
print("The resting ask remains:", stp_book.get_order(resting_ask["order_id"]))

Create a fresh book, then use limit() to rest an ask owned by client 7.
market('bid', client_id=7, ...): default self-trade prevention cancels the incoming buy.
{'avg_fill_price': None,
 'cancelled_quantity': 1,
 'client_id': 7,
 'evicted_orders': [],
 'filled_quantity': 0,
 'order_id': 100001,
 'order_type': 'market',
 'price': None,
 'replaced_order_id': None,
 'requested_quantity': 1,
 'resting_quantity': 0,
 'side': 'bid',
 'stp_cancelled_orders': [],
 'trades': []}
The resting ask remains: {'order_id': 100000, 'client_id': 7, 'side': 'ask', 'price': 10005, 'quantity': 2}


## What to remember

- Limit orders may rest; market orders do not.
- Price priority comes first; FIFO decides order among orders at the same price.
- Trade prices are the resting orders' prices.
- `best_bid`, `best_ask`, `mid_price`, and `spread` are properties, so read them without parentheses.
- Public methods such as `limit()`, `market()`, `modify_order()`, and `cancel_order()` return reports or IDs. Internal helpers such as `_submit()` and `_match()` implement the mechanics behind those calls.